# Delta Lake Optimization & Performance

OPTIMIZE, VACUUM, Z-ORDER, Liquid Clustering, Deletion Vectors, Predictive Optimization, and query plan analysis. Practical techniques for Delta table maintenance and performance.

**Prerequisites:** 02 — Delta Lake Advanced

## Learning Objectives

After completing this module you will be able to:

- **Diagnose** the small file problem and apply `OPTIMIZE` for compaction
- **Implement** partitioning strategies and understand when NOT to partition
- **Apply** Z-ORDER and Liquid Clustering for data skipping optimization
- **Manage** table lifecycle with `VACUUM` (retention, safety considerations)
- **Diagnose** data skew and apply salting or AQE to resolve it
- **Analyze** query plans with `EXPLAIN` to identify performance bottlenecks

## Setup

Environment configuration and database initialization for the optimization demos.

In [0]:
%run ../../setup/00_setup

### Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta

# Display user context
display(
    spark.createDataFrame([
        (CATALOG, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA)
    ], ['catalog', 'bronze_schema', 'silver_schema', 'gold_schema'])
)

# Set catalog and schema as default
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

## Optimization Techniques

File compaction (OPTIMIZE), data skipping (Z-ORDER), partitioning strategies, and Liquid Clustering — key techniques for query performance on large Delta tables.

**Theoretical Introduction:**

As data grows, query performance can degrade due to several factors:
- **Small Files Problem**: Too many small files increase metadata overhead
- **Data Layout**: Data not organized for common query patterns
- **Predicate Pushdown Inefficiency**: Scanning more data than necessary

Delta Lake provides several optimization techniques:

| Technique | Description | When to Use |
|-----------|-------------|-------------|
| **OPTIMIZE** | Compacts small files into larger ones | After many small writes |
| **Partitioning** | Physical data separation by column values | Low-cardinality filter columns |
| **Z-ORDER** | Co-locates related data for better pruning | Frequently filtered columns |
| **Liquid Clustering** | Modern alternative to partitioning + Z-ORDER | New tables (recommended) |

<img src="../../../assets/images/b784d6b9500944dcad315b7d75c5912b.webp" width="800">

**Optimization — Syntax Reference**

| Command | Syntax | Description |
|---------|--------|-------------|
| `OPTIMIZE` | `OPTIMIZE table` | Compact small files (target ~1 GB per file) |
| `OPTIMIZE + ZORDER` | `OPTIMIZE table ZORDER BY (col1, col2)` | Compact + co-locate data (max 4 cols) |
| `VACUUM` | `VACUUM table RETAIN 168 HOURS` | Remove unreferenced files (default: 7 days) |
| `VACUUM 0H` | `SET spark.databricks.delta.retentionDurationCheck.enabled = false; VACUUM t RETAIN 0 HOURS` | Removes ALL history |
| `DESCRIBE DETAIL` | `DESCRIBE DETAIL table` | Show file count, size, location, clustering |
| `DESCRIBE HISTORY` | `DESCRIBE HISTORY table` | Show all Delta versions with operation info |

### Example: The Small Files Problem

**Objective:** Demonstrate how many small files impact performance and how OPTIMIZE solves it

The "small files problem" occurs when:
- Streaming jobs write many small files
- Frequent small batch inserts
- High-concurrency writes

This leads to:
- Increased metadata overhead
- Slower query performance
- Higher storage costs (metadata per file)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.small_files_demo")

In [0]:
# Create a table with many small files (simulating streaming ingestion)
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.small_files_demo (
    id INT,
    data STRING,
    created_at TIMESTAMP
) USING DELTA
TBLPROPERTIES (
    delta.autoOptimize.optimizeWrite = false,
    delta.autoOptimize.autoCompact = false
)
""")

# Insert data in many small batches (simulating streaming)
from pyspark.sql.functions import lit, current_timestamp
import random
import string

print("Inserting 500 small batches to simulate streaming ingestion...")

In [0]:
%sql 

DESCRIBE DETAIL small_files_demo

In [0]:
from pyspark.sql.functions import lit, expr, current_timestamp
import random

# [1/2] Configure and generate DataFrame in memory (no Python loop — pure Spark)
total_files = 5000
rows_per_file = 2
total_rows = total_files * rows_per_file

df = (
    spark.range(0, total_rows)
    .withColumn("id", lit(random.randint(1, 100)))
    .withColumn("data", expr("uuid()"))
    .withColumn("created_at", current_timestamp())
)
print(f"DataFrame ready: {total_rows} rows → {total_files} small files planned")

In [0]:
# [2/2] Write with forced repartition — creates exactly total_files Delta files
df.repartition(total_files).write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.small_files_demo")

print(f"Done! Created {total_files} small files in a single transaction.")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.small_files_demo"))

In [0]:
# Check the number of files BEFORE optimization
before_optimize = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.small_files_demo")
display(before_optimize.select("numFiles", "sizeInBytes"))
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.small_files_demo"))

In [0]:
# Run OPTIMIZE to compact small files
optimize_result = spark.sql(f"""
    OPTIMIZE {CATALOG}.{BRONZE_SCHEMA}.small_files_demo
""")

display(optimize_result)

In [0]:
# ⚠ DANGER ZONE: VACUUM with 0 hours retention
# This disables the safety check and immediately deletes ALL old file versions.
# In production, NEVER use RETAIN 0 HOURS — use default 168 hours (7 days) or more!
# Files removed by VACUUM cannot be recovered. Time travel will stop working.
# We use this here ONLY because this is a training demo with disposable data.
spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled = false")

vacuum_result = spark.sql(f"""
    VACUUM {CATALOG}.{BRONZE_SCHEMA}.small_files_demo RETAIN 0 HOURS
""")

# Re-enable the safety check immediately
spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled = true")
print("VACUUM complete. Safety check re-enabled.")


In [0]:
# Check the number of files AFTER optimization
after_optimize = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.small_files_demo")
display(after_optimize.select("numFiles", "sizeInBytes"))

### Example: Partitioning

**Objective:** Demonstrate how partitioning improves query performance through partition pruning

| Syntax | Description |
|---|---|
| `PARTITIONED BY (col)` | Physically separates data into directories by column value |
| `PARTITIONED BY (col1, col2)` | Multi-column partitioning — each combination gets its own directory |
| `ALTER TABLE ... ADD PARTITION` | Adds a new partition to an existing table |
| `SHOW PARTITIONS table` | Lists all existing partitions |

Partitioning physically separates data into directories based on column values. This enables:
- **Partition Pruning**: Skip entire partitions that don't match query filters
- **Parallel Processing**: Process partitions independently

**Best Practices:**
- Use low-cardinality columns (date, country, status)
- Avoid over-partitioning (too many small partitions)
- Aim for 1GB+ per partition

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.orders_partitioned")

In [0]:
# Create a partitioned table
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.orders_partitioned (
    order_id STRING,
    customer_id STRING,
    product_id STRING,
    order_date DATE,
    amount DOUBLE,
    status STRING
) 
USING DELTA
PARTITIONED BY (order_date,status)
""")

In [0]:
# [1/2] Generate 30 days of orders data in memory (3,000 rows total)
from datetime import date, timedelta

orders_data = []
base_date = date(2024, 1, 1)

for day_offset in range(30):  # 30 days
    order_date = base_date + timedelta(days=day_offset)
    for i in range(100):  # 100 orders per day
        orders_data.append((
            f"ORD-{day_offset:02d}-{i:04d}",
            f"CUST{i % 50:04d}",
            f"PROD{i % 20:03d}",
            order_date,
            50 + (i * 2.5),
            "completed" if i % 3 != 0 else "pending"
        ))

print(f"Generated {len(orders_data)} order records across 30 days")

In [0]:
# [2/2] Create DataFrame and write — partitioned by order_date and status
orders_df = spark.createDataFrame(orders_data,
    ["order_id", "customer_id", "product_id", "order_date", "amount", "status"])

orders_df.write.format("delta").mode("append").partitionBy("order_date", "status") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.orders_partitioned")

print(f"Inserted {len(orders_data)} orders across 30 days")

In [0]:
# Check partitioning structure
display(spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.orders_partitioned"))

In [0]:
# Query with partition filter - only scans relevant partitions
# Check the Spark UI to see partition pruning in action
result = spark.sql(f"""
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.orders_partitioned
    WHERE order_date = '2024-01-15'
""")

print("Query for single date (should scan only 1 partition):")
display(result)

### Example: Z-ORDER (Data Skipping)

**Objective:** Demonstrate Z-ORDER for multi-dimensional clustering

| Command | Signature | Description |
|---|---|---|
| `OPTIMIZE ... ZORDER BY` | `OPTIMIZE table ZORDER BY (col1, col2)` | Compacts files AND re-orders data using Z-order curve for data skipping |
| `OPTIMIZE` (basic) | `OPTIMIZE table` | Compacts small files only — no Z-ORDER |

Z-ORDER is a multi-dimensional clustering technique that co-locates related data within files. This enables **data skipping** - reading only relevant files based on min/max statistics.

**When to Use:**
- Columns frequently in WHERE clauses
- High-cardinality columns (customer_id, product_id)
- Up to 4 columns (effectiveness decreases with more)

**How it Works:**
- Reorganizes data within files using Z-order curve
- Maintains min/max statistics per file
- Query engine skips files that don't match predicates

<img src="../../../assets/images/bccabee4ae7a4993b7e11321e8272261.webp" width="800">

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo")

In [0]:
# Create a table for Z-ORDER demonstration with auto-optimization disabled
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo (
    sale_id STRING,
    customer_id STRING,
    product_id STRING,
    store_id STRING,
    sale_date DATE,
    amount DOUBLE,
    quantity INT
) USING DELTA
TBLPROPERTIES (
    delta.autoOptimize.optimizeWrite = false,
    delta.autoOptimize.autoCompact = false
)
""")

In [0]:
# [1/2] Generate 100K sales records in memory
from datetime import date
import random

sales_data = []
for i in range(100000):  # 100K records
    sales_data.append((
        f"SALE-{i:08d}",
        f"CUST{random.randint(1, 1000):04d}",
        f"PROD{random.randint(1, 500):03d}",
        f"STORE{random.randint(1, 50):02d}",
        date(2024, random.randint(1, 12), random.randint(1, 28)),
        random.uniform(10, 500),
        random.randint(1, 10)
    ))

print(f"Generated {len(sales_data)} sales records")

In [0]:
# [2/2] Create DataFrame and write — auto-optimization disabled (we trigger manually)
sales_df = spark.createDataFrame(
    sales_data,
    ["sale_id", "customer_id", "product_id", "store_id", "sale_date", "amount", "quantity"]
)

sales_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo"
)
display(sales_df)

In [0]:
# Check file statistics BEFORE Z-ORDER
display(spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo"))

In [0]:
#Queries filtering before Z-Order
result = spark.sql(f"""
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo
    WHERE customer_id = 'CUST0393' AND product_id = 'PROD259' --CUST0592	PROD011
""")

display(result)

In [0]:
# Apply Z-ORDER on frequently filtered columns
# In this case: customer_id and product_id are common filter columns
zorder_result = spark.sql(f"""
    OPTIMIZE {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo
    ZORDER BY (customer_id, product_id)
""")

display(zorder_result)

In [0]:
# Example query that benefits from Z-ORDER
result = spark.sql(f"""
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo
     WHERE customer_id = 'CUST0393' AND product_id = 'PROD259' --CUST0592	PROD011
""")

print("Query with Z-ORDER optimized columns (check Spark UI for data skipping):")
display(result)

### Example: Liquid Clustering

**Objective:** Demonstrate Liquid Clustering as a modern alternative to partitioning and Z-ORDER

| Syntax | Description |
|---|---|
| `CLUSTER BY (col1, col2)` | Defines clustering columns at table creation |
| `CLUSTER BY AUTO` | Databricks automatically selects optimal clustering columns |
| `ALTER TABLE ... CLUSTER BY (col)` | Changes clustering columns without rewriting data |
| `OPTIMIZE table` | Applies Liquid Clustering incrementally (no ZORDER needed) |

Liquid Clustering is Databricks' latest optimization technique that combines the benefits of partitioning and Z-ORDER while being easier to manage:

**Key Benefits:**
- **Automatic**: Databricks manages data layout automatically
- **Adaptive**: Adjusts to changing query patterns over time
- **Flexible**: Can change clustering columns without rewriting data
- **Incremental**: Works incrementally with each OPTIMIZE
- **Simpler**: No need to choose between partitioning and Z-ORDER

**When to Use:**
- New tables (recommended default)
- Tables with evolving query patterns
- When you're unsure about optimal partitioning strategy
**Automatic Liquid Clustering (GA June 2025):**
- Use `CLUSTER BY AUTO` — Databricks automatically selects the best clustering columns based on query patterns
- No need to manually choose columns — the system learns from workload
- Combine with Predictive Optimization for fully automated table maintenance

**Automatic Liquid Clustering (GA June 2025):**
- Use `CLUSTER BY AUTO` — Databricks automatically selects the best clustering columns based on query patterns
- No need to manually choose columns — the system learns from workload
- Combine with Predictive Optimization for fully automated table maintenance

**Automatic Liquid Clustering (GA June 2025):**
- Use `CLUSTER BY AUTO` — Databricks automatically selects the best clustering columns based on query patterns
- No need to manually choose columns — the system learns from workload
- Combine with Predictive Optimization for fully automated table maintenance

**Liquid Clustering — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Create with clustering | `CREATE TABLE t (...) CLUSTER BY (col1, col2)` |
| Add to existing table | `ALTER TABLE t CLUSTER BY (col1, col2)` |
| Change columns | `ALTER TABLE t CLUSTER BY (col3)` — no data rewrite needed |
| Remove clustering | `ALTER TABLE t CLUSTER BY NONE` |
| Trigger compaction | `OPTIMIZE t` — applies liquid clustering automatically |
| Check clustering | `DESCRIBE DETAIL t` — see `clusteringColumns` field |

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering")

In [0]:
# Create a table with Liquid Clustering, auto-optimization disabled, and Predictive Optimization enabled
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering (
    sale_id STRING,
    customer_id STRING,
    product_id STRING,
    region STRING,
    sale_date DATE,
    amount DOUBLE,
    quantity INT
) 
USING DELTA
CLUSTER BY (customer_id, region)
TBLPROPERTIES (
    delta.autoOptimize.optimizeWrite = false,
    delta.autoOptimize.autoCompact = false
)
""")

In [0]:
# Enable Predictive Optimization — correct syntax (DBR 14+)
spark.sql(f"""ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering
  ENABLE PREDICTIVE OPTIMIZATION""")
print("Predictive Optimization enabled for sales_liquid_clustering")

In [0]:
from pyspark.sql.functions import col, rand, lit, concat, lpad, element_at, array, date_add, to_date, round

# [1/2] Configure and generate DataFrame in memory (vectorized — no Python loop)
target_files = 5000
rows_per_file = 10
total_rows = target_files * rows_per_file

regions_list = array([lit(x) for x in ['North', 'South', 'East', 'West', 'Central']])

df = spark.range(0, total_rows).withColumnRenamed("id", "idx") \
    .withColumn("sale_id", concat(lit("SALE-"), lpad(col("idx"), 8, "0"))) \
    .withColumn("customer_id", concat(lit("CUST"), lpad((rand() * 500 + 1).cast("int"), 4, "0"))) \
    .withColumn("product_id", concat(lit("PROD"), lpad((rand() * 200 + 1).cast("int"), 3, "0"))) \
    .withColumn("region", element_at(regions_list, (rand() * 5 + 1).cast("int"))) \
    .withColumn("sale_date", date_add(to_date(lit("2024-01-01")), (rand() * 364).cast("int"))) \
    .withColumn("amount", round(rand() * 490 + 10, 2)) \
    .withColumn("quantity", (rand() * 10 + 1).cast("int")) \
    .drop("idx")

print(f"DataFrame ready: {total_rows} rows → {target_files} small files planned")

In [0]:
# [2/2] Write — repartition forces file count to simulate fragmented table
df.repartition(target_files).write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering")

print(f"Done! Inserted {total_rows} records in {target_files} small files.")

In [0]:
df = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering")
display(df)

In [0]:
# Change clustering columns on an existing table (no data rewrite needed!)
# CLUSTER BY AUTO lets Databricks automatically choose optimal clustering columns
spark.sql(f"""
ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering
CLUSTER BY AUTO
""")

In [0]:
# OPTIMIZE automatically applies Liquid Clustering
# No need to specify ZORDER - it's built into the table definition!
optimize_result = spark.sql(f"""
    OPTIMIZE {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering
""")

display(optimize_result)

In [0]:
# Check clustering information
display(spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering"))

In [0]:
# Queries filtering by clustering columns are automatically optimized
result = spark.sql(f"""
    SELECT region, COUNT(*) as sales_count, SUM(amount) as total_amount
    FROM {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering
    WHERE customer_id LIKE 'CUST00%' AND region = 'North'
    GROUP BY region
""")

display(result)

**Comparison: Partitioning vs Z-ORDER vs Liquid Clustering**

| Feature | Partitioning | Z-ORDER | Liquid Clustering |
|---------|-------------|---------|-------------------|
| When to choose | Low-cardinality columns | High-cardinality filter columns | General purpose (recommended) |
| Data layout | Directory per partition | Co-located in files | Automatic clustering |
| Schema change | Requires rewrite | Easy to change | Easy to change |
| Maintenance | Manual | Manual OPTIMIZE | Automatic with OPTIMIZE |
| Best for | Date/Region filters | Multi-column filters | Evolving workloads |

## Query Plan Analysis with EXPLAIN

Use `EXPLAIN` to understand how Spark executes a query — identify full scans, shuffles, and data skipping.

| Command | Description |
|---|---|
| `EXPLAIN query` | Basic physical plan |
| `EXPLAIN FORMATTED query` | Detailed plan with metrics |
| `EXPLAIN EXTENDED query` | Logical + physical + optimized plan |

**What to look for:**
- [OK] **PartitionFilters** — partition pruning is active
- [OK] **DataFilters / PushedFilters** — file-level data skipping
- [OK] **BroadcastHashJoin** — small table broadcast (no shuffle)
- [!] **SortMergeJoin** — both sides shuffled (expensive)
- [!] **Scan** without filters — full table scan

In [0]:
# EXPLAIN on a Z-ORDERed table (should show DataFilters / data skipping)
spark.sql(f"""
    EXPLAIN FORMATTED
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo
    WHERE customer_id = 'CUST0393' AND product_id = 'PROD259'
""").display(truncate=False)

In [0]:
# Compare: EXPLAIN on a Liquid Clustered table
spark.sql(f"""
    EXPLAIN FORMATTED
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering
    WHERE product_id = 'PROD042'
""").display(truncate=False)

In [0]:
# EXPLAIN with JOIN — check for BroadcastHashJoin vs SortMergeJoin
spark.sql(f"""
    EXPLAIN FORMATTED
    SELECT o.*, c.first_name, c.last_name
    FROM {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo o
    JOIN {CATALOG}.{BRONZE_SCHEMA}.customers c
      ON o.customer_id = c.customer_id
    WHERE o.product_id = 'PROD259'
""").display(truncate=False)

## Deletion Vectors

Faster DELETE, UPDATE, and MERGE operations by marking rows as deleted in a separate bitmap file instead of rewriting entire Parquet files.

**Theoretical Introduction:**

Deletion Vectors are a storage optimization feature in Delta Lake that improves DELETE, UPDATE, and MERGE performance.

**How it works:**
- Instead of rewriting entire data files on DELETE/UPDATE, Delta Lake marks rows as deleted in a separate **deletion vector file**
- The actual data files remain unchanged until next OPTIMIZE or REORG
- Reads automatically filter out deleted rows using the deletion vector

<img src="../../../assets/images/42198416fc184ab18e07b5c0bcd29468.webp" width="800">

| Aspect | Without Deletion Vectors | With Deletion Vectors |
|--------|------------------------|----------------------|
| DELETE speed | Slow (rewrite files) | Fast (mark in vector) |
| Storage during DELETE | Temporary 2x | Minimal overhead |
| Read performance | Normal | Slight overhead (filter) |
| Cleanup | Automatic | Needs REORG TABLE ... APPLY |

```sql
-- Enable deletion vectors on a table
ALTER TABLE my_table SET TBLPROPERTIES ('delta.enableDeletionVectors' = true);

-- Purge deletion vectors (rewrite files)
REORG TABLE my_table APPLY (PURGE);
```

**Pro Tip:** Deletion Vectors are enabled by default on new tables in Databricks. They make DML operations faster but may slightly increase read overhead until REORG is run.

## Predictive Optimization

Automatic maintenance feature that uses ML to schedule OPTIMIZE and VACUUM — eliminating manual table maintenance for Unity Catalog managed tables.

![ae9f6138b2354c91a096cfd314482a](../../../assets/images/2250446af43249ff80cc2eb86cdf8816.webp "../../../assets/images/2250446af43249ff80cc2eb86cdf8816.webp")

**Key Facts (2025-2026):**

- **Enabled by default** since May 2025 for all Unity Catalog managed tables
- Automatically runs OPTIMIZE (file compaction) when small files accumulate
- Automatically runs VACUUM when orphaned files need cleanup
- Managed at catalog, schema, or table level
- No manual scheduling needed — Databricks decides when to run based on table health metrics

```sql
-- Enable at schema level (already default for new schemas since May 2025)
ALTER SCHEMA my_catalog.my_schema
ENABLE PREDICTIVE OPTIMIZATION;

-- Disable for a specific table (e.g., staging tables)
ALTER TABLE my_catalog.my_schema.staging_temp
DISABLE PREDICTIVE OPTIMIZATION;

-- Check optimization history
SELECT * FROM system.storage.predictive_optimization_operations_history
WHERE catalog_name = 'my_catalog'
ORDER BY timestamp DESC;
```

| Feature | Manual Maintenance | Predictive Optimization |
|---------|-------------------|------------------------|
| OPTIMIZE | Run manually / schedule | Automatic |
| VACUUM | Run manually / schedule | Automatic |
| Tuning | DBA responsibility | ML-based decisions |
| Cost | Compute cost on schedule | Pay per optimization |
| Default | Must configure | Enabled by default (May 2025+) |

> **Pro Tip:** Predictive Optimization requires Unity Catalog managed tables. For staging/temporary tables that are dropped frequently, consider disabling it to avoid unnecessary optimization runs.

## Performance Bottlenecks (Optional Demo)

> **Trainer note:** This section is optional — cover it if time allows. It maps to DEA Exam Domain 6 (Troubleshooting, Monitoring, and Optimization).


### 2.1 Data Skew

**Data skew** occurs when data is distributed unevenly across partitions. One executor ends up processing 10× or 100× more rows than others — this single "straggler" task determines the total stage duration.

**Why it matters:**
- Spark runs all tasks in parallel, but **waits for the slowest task** to finish the stage
- 1 skewed partition → all other executors sit idle
- Typical symptom: a stage completes in seconds, except one task still running for minutes

**Common causes of skew:**
- Joining or grouping on a column with a **dominant value** (e.g., a top customer, `NULL`, a single popular product)
- Low-cardinality groupBy keys
- Badly chosen partition columns

#### How to Detect Data Skew in Spark UI

**Step-by-step walkthrough:**

1. Open **Spark UI** (cluster → Spark UI, or click the Spark UI link in a running job)
2. Go to **Jobs** tab → click the job that ran slowly
3. Open **Stages** tab → click the stage with the longest duration
4. Scroll to **Summary Metrics** section and look at:
   - **Duration** row: compare `Min`, `Median`, `Max`
   - **Input Size** row: compare `Min` vs `Max` per task
5. Scroll to the **Tasks** table → sort by **Duration** descending

**What healthy vs skewed looks like:**

| Metric | Healthy | Skewed |
|--------|---------|--------|
| Max task duration | ≈ Median | >> Median (3× or more) |
| Task duration histogram | Tight cluster | Long right tail |
| Input size per task | Roughly equal across tasks | 1–2 tasks have far more bytes |
| GC Time | Low | Elevated on the skewed task |

> **Databricks tip:** The Stage detail page shows a **task timeline** — a skewed stage has one very long bar on the right while all others are short. The **Event Timeline** view makes this immediately visible.

In [0]:
# [1/4] Setup — imports and skewed dataset creation
# Disable AQE so the skew is visible in Spark UI (AQE would fix it automatically)
spark.conf.set("spark.sql.adaptive.enabled", "false")

# Scenario: customer_id=99 is one large enterprise — generates 90% of all orders
import pyspark.sql.functions as F
from pyspark.sql import Row

skewed_data = (
    [Row(customer_id=99, order_id=i, amount=float(i % 500)) for i in range(900000)] +
    [Row(customer_id=i % 50, order_id=9000 + i, amount=float(i % 200)) for i in range(100000)]
)
df_skewed = spark.createDataFrame(skewed_data)
print(f"Total rows: {df_skewed.count()}")

In [0]:
# [2/4] Inspect key distribution — see the skew before anything runs
print("=== Row counts per customer_id ===")
df_skewed.groupBy("customer_id").count().orderBy(F.desc("count")).display()
# customer_id=99   (99%)
# all others       (1% total)
# This is the skewed key — joining or groupBy on it will create a straggler task

In [0]:
# [3/4] Repartition by skewed key — show uneven partition sizes
df_partitioned = df_skewed.repartition(20, "customer_id")

print("=== Rows per partition (repartitioned by customer_id) ===")
partition_counts = (
    df_partitioned.withColumn("_pid", F.spark_partition_id())
    .groupBy("_pid").count().orderBy("_pid").display()
)

# One partition holds ~9000 rows, others ~100 or 0
# In a real cluster this maps to ONE task doing all the work

In [0]:
# [4/4] Trigger a real job — then inspect it in Spark UI
df_partitioned.groupBy("customer_id").agg(F.sum("amount")).display()

print("Job finished. Now open Spark UI to see the skew:")
print("  1. Click 'Spark UI' in the cluster panel (or notebook header link)")
print("  2. Jobs tab -> click the most recent job")
print("  3. Stages tab -> click the stage with the most tasks")
print("  4. Scroll to 'Summary Metrics':")
print("     - Duration  -> if Max >> Median: SKEW CONFIRMED")
print("     - Input Size -> skewed task reads far more bytes than others")

#### How to Fix Data Skew

**Option 1 — AQE Skew Join (recommended, automatic):**
AQE automatically splits oversized partitions during sort-merge joins. Enabled by default in Databricks Runtime 9+. No code change needed — verify it's on:
```python
spark.conf.get("spark.sql.adaptive.skewJoin.enabled")  # should be "true"
```

**Option 2 — Salting (manual, for groupBy or non-AQE joins):**
Append a random number to the skewed key before the operation, then strip it after.

**Steps:**
1. Add salt: `key + '_' + random(0..SALT_FACTOR-1)`
2. groupBy on the salted key
3. Re-aggregate to collapse the salted sub-groups back into the original key

**Option 3 — Repartition before heavy operation:**
```python
df.repartition(200)              # full shuffle — rebalances all partitions
df.repartition(200, "some_col") # shuffle by column — deterministic distribution
```

| Technique | When to use | Complexity |
|-----------|------------|------------|
| **AQE skew join** | Sort-merge join on skewed key, DBR 9+ | Zero (automatic) |
| **Salting** | groupBy or join where AQE does not help | Medium |
| **Repartition(n)** | Even spread before a heavy op, no dominant value | Low |
| **Filter dominant key** | Process the dominant value separately, then union back | Medium |

In [0]:
# [Fix 1/4] Re-enable AQE and verify — handles skew joins automatically in DBR 9+
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("AQE enabled:      ", spark.conf.get("spark.sql.adaptive.enabled"))
print("AQE skew join:    ", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
print("Skew factor:      ", spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionFactor"))
print("Skew threshold:   ", spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes"))
# If skew join = true -> sort-merge joins are handled automatically
# For groupBy skew (no join involved) -> AQE does NOT help -> use salting

In [0]:
# [Fix 2/4] Add salt — break the dominant key into sub-keys
SALT_FACTOR = 10  # customer_id=99 splits into "99_0", "99_1", ..., "99_9"

df_salted = df_skewed.withColumn(
    "customer_id_salted",
    F.concat(
        F.col("customer_id").cast("string"),
        F.lit("_"),
        (F.rand() * SALT_FACTOR).cast("int").cast("string")
    )
)

print("=== Sample: customer_id=99 keys after salting ===")
(df_salted
 .filter(F.col("customer_id") == 99)
 .select("customer_id", "customer_id_salted")
 .distinct()
 .orderBy("customer_id_salted")
 .display())

In [0]:
# [Fix 3/4] Repartition on salted key — compare partition sizes to [3/4] above
df_fixed = df_salted.repartition(10, "customer_id_salted")

print("=== Rows per partition AFTER salting ===")
partition_counts_fixed = (
    df_fixed.withColumn("_pid", F.spark_partition_id())
    .groupBy("_pid").count().orderBy("_pid").collect()
)
for row in partition_counts_fixed:

    print(f"  Partition {row['_pid']:2d}: {row['count']:5d} rows ")
# All partitions should now be roughly equal (~1000 rows each)
# Compare to Cell [3/4]: from one overloaded partition to balanced distribution

In [0]:
# [Fix 4/4] Double aggregation — aggregate on salted key, then strip salt and re-aggregate
df_aggregated = (
    df_salted
    # First groupBy: on salted key — each sub-partition runs in parallel
    .groupBy("customer_id_salted")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("order_count")
    )
    # Strip the salt suffix to recover the original customer_id
    .withColumn("customer_id", F.split(F.col("customer_id_salted"), "_")[0].cast("int"))
    # Second groupBy: collapse sub-groups back into one row per customer
    .groupBy("customer_id")
    .agg(
        F.sum("total_amount").alias("total_amount"),
        F.sum("order_count").alias("order_count")
    )
)

print("=== Final aggregated result (salt stripped) ===")
df_aggregated.orderBy(F.desc("order_count")).display()

print("\nNow check Spark UI again — compare task durations to the job in [4/4]")
print("Max Duration should be close to Median: skew resolved.")

### 2. Excessive Shuffle

A **shuffle** is triggered by any operation that requires redistributing data across executors:

| Operation | Shuffle triggered? |
|-----------|-----------------|
| `groupBy` + aggregation | Yes |
| `join` (sort-merge) | Yes |
| `distinct` | Yes |
| `repartition(n)` | Yes |
| `coalesce(n)` | No (narrow) |
| `filter`, `select`, `withColumn` | No (narrow) |
| broadcast join | No |

**How to reduce shuffle:**

1. **Broadcast join** — if one table is small, force a broadcast to avoid shuffle entirely
2. **Partition pruning** — filter early to reduce rows before a join
3. **Coalesce vs repartition**:
   - `coalesce(n)` — reduces partitions without a full shuffle (use to shrink before write)
   - `repartition(n)` — full shuffle (use to increase partitions or rebalance before heavy ops)


In [0]:
# Broadcast join: avoid shuffle for small dimension tables
import pyspark.sql.functions as F
from pyspark.sql.functions import broadcast

orders = (spark.range(1_000_000)
          .withColumnRenamed('id', 'order_id')
          .withColumn('product_id', (F.col('order_id') % 100).cast('int')))

products = (spark.range(100)
            .withColumnRenamed('id', 'product_id')
            .withColumn('product_name',
                        F.concat(F.lit('Product_'), F.col('product_id'))))

# With broadcast hint: Spark broadcasts the small table to all executors
# No shuffle on the large table side
result = orders.join(broadcast(products), on='product_id')
print('Join plan (look for BroadcastHashJoin):')
result.explain(mode='formatted')


### 3. Disk Spilling

**Spilling** occurs when Spark cannot fit shuffle data or sort buffers in executor memory and writes the overflow to disk.

**Causes:**
- Too few partitions — each partition is too large to fit in memory
- Executor memory too small for the workload
- Large sort operations (e.g., `orderBy` on a huge dataset)

**How to detect spilling:**
- Spark UI → **Stages** tab → click a stage → scroll to the **Summary Metrics** table
- Look for **Spill (Memory)** and **Spill (Disk)** columns — any non-zero value indicates spilling

**How to fix spilling:**

| Fix | How |
|-----|-----|
| Increase `spark.sql.shuffle.partitions` | Smaller partitions fit in memory |
| Enable AQE coalesce (on by default) | AQE merges small partitions after shuffle |
| Increase executor memory | Resize cluster node type |
| Reduce broadcast threshold | Avoid accidental large broadcasts |


In [0]:
# Adjusting shuffle partitions for the current session
print('Current shuffle partitions:', spark.conf.get('spark.sql.shuffle.partitions'))

# For a large dataset, increase partition count to reduce per-partition size
spark.conf.set('spark.sql.shuffle.partitions', '400')
print('Updated shuffle partitions:', spark.conf.get('spark.sql.shuffle.partitions'))

# Reset to default for the rest of the notebook
spark.conf.set('spark.sql.shuffle.partitions', '200')
print('Reset to:', spark.conf.get('spark.sql.shuffle.partitions'))


### 4. Adaptive Query Execution (AQE)

**Adaptive Query Execution** (AQE) is a Spark optimization framework that re-optimizes query plans at runtime, using statistics collected during execution.

AQE is **enabled by default** in Databricks Runtime 9.0+ (`spark.sql.adaptive.enabled = true`).

| Feature | Config flag | What it does |
|---------|------------|-------------- |
| **Coalescing small partitions** | `spark.sql.adaptive.coalescePartitions.enabled` (default: true) | After shuffle, merges small partitions to reduce task overhead |
| **Skew join optimization** | `spark.sql.adaptive.skewJoin.enabled` (default: true) | Splits oversized partitions and replicates the matching side to fix skew |
| **Dynamic broadcast threshold** | `spark.sql.adaptive.localShuffleReader.enabled` | Upgrades a sort-merge join to a broadcast join at runtime if one side is small enough |

**How to verify AQE is active:** run a join/aggregation, open Spark UI → SQL tab, look for `AQEShuffleRead` nodes or `BroadcastHashJoin` in the physical plan.


In [0]:
# Verify AQE configuration
aqe_configs = {
    'spark.sql.adaptive.enabled': 'Master AQE switch',
    'spark.sql.adaptive.coalescePartitions.enabled': 'Merge small post-shuffle partitions',
    'spark.sql.adaptive.skewJoin.enabled': 'Split skewed partitions in joins',
    'spark.sql.adaptive.skewJoin.skewedPartitionFactor': 'Partition is skewed if size > factor * median',
    'spark.sql.autoBroadcastJoinThreshold': 'Threshold for automatic broadcast join',
    'spark.sql.shuffle.partitions': 'Default number of shuffle partitions',
}


## Summary

| Topic | Key Takeaway |
|---|---|
| **OPTIMIZE** | Compacts small files. Run after streaming/frequent inserts |
| **Partitioning** | Low-cardinality columns, partition > 1GB |
| **Z-ORDER** | Co-locates data for data skipping on filter columns |
| **Liquid Clustering** | Modern replacement — incremental, OPTIMIZE still needed but faster |
| **Data Skew** | AQE handles most cases. Salting, broadcast for extreme skew |
| **Deletion Vectors** | Fast DELETE/UPDATE via marking instead of rewriting files |
| **Predictive Optimization** | Automatic OPTIMIZE + VACUUM for UC managed tables |

### Quick Reference

| Operation | Command |
|---|---|
| Optimize | `OPTIMIZE table` |
| Z-ORDER | `OPTIMIZE table ZORDER BY (col)` |
| Vacuum | `VACUUM table RETAIN X HOURS` |
| Deletion Vectors | `ALTER TABLE SET TBLPROPERTIES ('delta.enableDeletionVectors' = true)` |
| Purge DVs | `REORG TABLE ... APPLY (PURGE)` |

> **Note:** Change Data Feed (CDF) has been moved to **M03** (fundamentals) and **M05** (incremental ETL usage).

## Cleanup

Remove demo tables and temporary data created during this module.

In [0]:
# Optional test resource cleanup
# NOTE: Run only if you want to delete all created data

# Tables to clean up:
cleanup_tables = [
    "small_files_demo",
    "sales_zorder_demo",
    "sales_liquid_clustering"
]

In [0]:
# Uncomment below to execute cleanup:
# for table in cleanup_tables:
#     spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.{table}")
#     print(f"Dropped: {table}")

# spark.sql("DROP VIEW IF EXISTS customer_updates")
# spark.catalog.clearCache()

# print("All resources cleaned up!")

← [03 — Delta Lake](../../day1/demo/03_delta_lake.ipynb) | **[ README](../../../README.md)** | [05 — Incremental Ingestion →](05_incremental_ingestion.ipynb)